In [1]:
from preprocessing import preprocess_data
from feature_engineering import run_feature_engineering
from make_model import train_and_evaluate
from feature_importance import analyze_feature_importance
from inference import predict_score
from verify_training_data import diagnose_training_data

### Preprocessing
Raw data from multiple rides with the earable is combined into a master `.csv` per sensors, e.g. `master_gyroscope.csv`.

In [2]:
preprocess_data()

Starting data preprocessing...
Saved Labels: ./processed/y_labels.csv (12 rides labeled)
Saved Sensor Data: ./processed/master_temperature_sensor.csv (19825 total rows across all rides)
Saved Sensor Data: ./processed/master_optical_temperature_sensor.csv (24662 total rows across all rides)
Saved Sensor Data: ./processed/master_barometer.csv (19825 total rows across all rides)
Saved Sensor Data: ./processed/master_gyroscope.csv (316897 total rows across all rides)
Saved Sensor Data: ./processed/master_magnetometer.csv (316885 total rows across all rides)
Saved Sensor Data: ./processed/master_accelerometer.csv (316909 total rows across all rides)
Saved Sensor Data: ./processed/master_photoplethysmography.csv (158649 total rows across all rides)

Preprocessing complete, data is ready for tsfresh.


### Feature engineering using tsfresh
The master `.csv` is processed by tsfresh to extract features such as maximum value, standard deviation, median, etc. which can be processed by XGBoost. Exclude sensors by putting them in the `exclude_sensor` list.

In [ ]:
# exclude_sensors = ["optical_temperature_sensor", "accelerometer", "gyroscope", "magnetometer", "barometer", "photoplethysmography", "temperature_sensor"]
exclude_sensors = ["optical_temperature_sensor", "temperature_sensor"]
run_feature_engineering(exclude_sensors=exclude_sensors)

Starting tsfresh feature engineering...

Merging data...


ValueError: No objects to concatenate

### Train and evaluate model using XGBoost

In [7]:
train_and_evaluate()

Loading data for training...
Dataset ready: 12 rides, 160 features per ride.
Evaluating model using Leave-One-Out Cross-Validation...

MODEL PERFORMANCE REPORT
Mean Absolute Error (MAE): 0.73
Root Mean Squared Error (RMSE): 0.79
(Interpretation: The model's guesses are off by an average of 0.73 points on the 0-4 scale)

Training final production model on 100% of the data...
SUCCESS! Final trained model saved to: ./models/model.json


### Evaluate feature importance of model
XGBoost has built-in functions to show the importance per feature.

In [6]:
analyze_feature_importance()


Top 15 Most Important Features for Predicting Distraction:
----------------------------------------------------------------------
Rank  | Feature Name                                  | Importance Score
----------------------------------------------------------------------
1     | OPTICAL_TEMPERATURE_SENSOR_Temperature__sum_values | 0.5002
2     | BAROMETER_Pressure__sum_values                | 0.3521
3     | BAROMETER_Pressure__standard_deviation        | 0.0906
4     | BAROMETER_Pressure__minimum                   | 0.0249
5     | MAGNETOMETER_X__sum_values                    | 0.0138
6     | OPTICAL_TEMPERATURE_SENSOR_Temperature__standard_deviation | 0.0104
7     | BAROMETER_Pressure__median                    | 0.0052
8     | PHOTOPLETHYSMOGRAPHY_RED__sum_values          | 0.0028
----------------------------------------------------------------------

Full importance ranking saved to: ./models/feature_importance.csv


### Run inference on a single ride
To run inference on a single ride (folder), give `new_ride_folder="..."` as an argument and the model will give a distraction score.

In [ ]:
predict_score(new_ride_folder="2024-06-17_16-00-00")

Processing new ride: OpenWearable_Recording_2026-03-26T101935.303786_3
Calculated distraction score: 2.35 / 4.0


### Check scores for all the rides in the training data
Gives a list of rides used in training the model and their scores according to the model

In [ ]:
diagnose_training_data()

Loading data and model for diagnostics...

Ride ID (Shortened)       | True Score | Predicted  | Error     
-----------------------------------------------------------------
2026-03-26T101341.966067  | 4.0        | 3.70       | 0.30      
2026-03-26T101935.303786  | 3.0        | 2.89       | 0.11      
2026-03-26T102512.493306  | 2.0        | 2.05       | 0.05      
2026-03-26T103015.953397  | 1.0        | 1.19       | 0.19      
2026-03-26T103513.282691  | 0.0        | 0.53       | 0.53      
2026-03-26T105712.346697  | 4.0        | 3.62       | 0.38      
2026-03-26T110257.381741  | 3.0        | 2.85       | 0.15      
2026-03-26T110809.275003  | 2.0        | 2.09       | 0.09      
2026-03-26T111245.688277  | 1.0        | 1.09       | 0.09      
2026-03-26T111650.158656  | 0.0        | 0.52       | 0.52      
2026-03-26T112950.696222  | 4.0        | 3.61       | 0.39      
2026-03-26T113555.625881  | 3.0        | 2.96       | 0.04      
----------------------------------------------